# EzLab — Kaggle LoRA → GGUF (Unsloth only)

Fine-tune **Qwen2.5-1.5B-Instruct** with **Unsloth** on the task/code/output schema, then export **GGUF Q4_K_M** with Unsloth (no Hugging Face login, no PEFT merge fallback, no llama.cpp).

```json
{"code": "...", "output": "..."}
```

## Setup
1. **GPU** on (T4/P100) + **Internet on** (Unsloth pulls the base model once)
2. **Add Data** → your `kaggle_dataset.zip` (`train.jsonl` + `valid.jsonl`)
3. **Run All** — leave the tab open (~45–90 min)
4. Download `/kaggle/working/labpulse/gguf/labpulse-1.5b-q4_k_m.gguf` **before** stopping the session
5. Optional: **Save Version** so Output survives if the session dies
6. On Mac: `mv` into `Labpulse/models/labpulse-1.5b-q4_k_m.gguf` and restart the API


## 0) Config


In [ ]:
from pathlib import Path
import os

# Unsloth only — no HF login / token required for public Unsloth models
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
# Do not force Hugging Face auth
for k in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACE_HUB_TOKEN"):
    os.environ.pop(k, None)

# 1.5B fits free T4. Set USE_3B=True only if you have enough VRAM.
USE_3B = False
MODEL_ID = "unsloth/Qwen2.5-3B-Instruct" if USE_3B else "unsloth/Qwen2.5-1.5B-Instruct"

MAX_SEQ_LENGTH = 4096
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
EPOCHS = 3
LR = 1.5e-4
BATCH_SIZE = 1 if USE_3B else 2
GRAD_ACCUM = 8 if USE_3B else 4
SEED = 42
WARMUP_RATIO = 0.05

WORK = Path("/kaggle/working/labpulse")
DATA_DIR = WORK / "data"
ADAPTER_DIR = WORK / "adapters"
MERGED_DIR = WORK / "merged_16bit"
GGUF_DIR = WORK / "gguf"
GGUF_NAME = "labpulse-3b-q4_k_m.gguf" if USE_3B else "labpulse-1.5b-q4_k_m.gguf"
GGUF_OUT = GGUF_DIR / GGUF_NAME

for p in (WORK, DATA_DIR, ADAPTER_DIR, MERGED_DIR, GGUF_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("WORK =", WORK)
print("MODEL_ID =", MODEL_ID)
print("GGUF_OUT =", GGUF_OUT)
print("Stack: Unsloth only (merge + GGUF). Internet must be ON for first model download.")


## 1) GPU check + install deps


In [ ]:
import subprocess, sys

def sh(*args):
    print("+", " ".join(args))
    subprocess.check_call(list(args))

# Fail fast if no GPU (do NOT use SystemExit — it confuses Kaggle/IPython)
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected.\n"
        "Fix on Kaggle:\n"
        "  1. Right sidebar → Settings\n"
        "  2. Accelerator → GPU (T4 or P100)\n"
        "  3. Wait for session restart → Run All\n"
    )

print("CUDA device:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# Unsloth pulls its own stack — install Unsloth first, then TRL/datasets for SFT
sh(sys.executable, "-m", "pip", "install", "-q", "-U", "pip")
sh(sys.executable, "-m", "pip", "install", "-q", "unsloth")
sh(sys.executable, "-m", "pip", "install", "-q", "datasets", "trl")
print("deps OK (Unsloth)")


## 2) Load dataset

**Option A (recommended):** Add your Kaggle Dataset that contains `train.jsonl` + `valid.jsonl`  
(from `Labpulse/data/kaggle_dataset.zip`).

**Option B:** Upload `kaggle_dataset.zip` into this notebook’s working directory.


In [ ]:
import json, shutil, zipfile
from pathlib import Path

DATA_DIR = Path("/kaggle/working/labpulse/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_path = DATA_DIR / "train.jsonl"
valid_path = DATA_DIR / "valid.jsonl"

def find_jsonl(name: str) -> Path | None:
    # Search Kaggle input mounts + working dir
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob(name):
            if p.is_file():
                return p
    return None

def extract_zip(zpath: Path) -> None:
    print("Extracting", zpath)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(DATA_DIR)

# Prefer already-present jsonl, else zip
src_train = find_jsonl("train.jsonl")
src_valid = find_jsonl("valid.jsonl")
if src_train and src_valid:
    if src_train.resolve() != train_path.resolve():
        shutil.copy2(src_train, train_path)
    if src_valid.resolve() != valid_path.resolve():
        shutil.copy2(src_valid, valid_path)
else:
    zips = list(Path("/kaggle/input").rglob("*.zip")) if Path("/kaggle/input").exists() else []
    zips += list(Path("/kaggle/working").glob("*.zip"))
    zips += list(Path(".").glob("*dataset*.zip"))
    if not zips:
        raise FileNotFoundError(
            "Add kaggle_dataset.zip as a Dataset (Add Data) or upload it to /kaggle/working. "
            "It must contain train.jsonl and valid.jsonl."
        )
    extract_zip(zips[0])

assert train_path.exists() and valid_path.exists(), (train_path, valid_path)
print("train lines", sum(1 for _ in train_path.open()))
print("valid lines", sum(1 for _ in valid_path.open()))


In [ ]:
from datasets import Dataset

REQUIRED_ASSISTANT_KEYS = {"code", "output"}

def load_messages(path: Path):
    rows = []
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            obj = json.loads(line)
            if "messages" not in obj:
                raise ValueError(f"{path.name}:{i} missing messages[]")
            msgs = obj["messages"]
            if len(msgs) < 3 or msgs[-1]["role"] != "assistant":
                raise ValueError(f"{path.name}:{i} bad roles")
            payload = json.loads(msgs[-1]["content"])
            if not REQUIRED_ASSISTANT_KEYS.issubset(payload):
                raise ValueError(f"{path.name}:{i} assistant missing code/output: {payload.keys()}")
            if not str(payload["code"]).strip() or "placeholder" in str(payload["code"]).lower():
                raise ValueError(f"{path.name}:{i} weak code")
            rows.append({"messages": msgs})
    return rows

train_rows = load_messages(train_path)
valid_rows = load_messages(valid_path)
print("loaded", len(train_rows), "train /", len(valid_rows), "valid")
print("sample roles:", [m["role"] for m in train_rows[0]["messages"]])
print("assistant keys:", sorted(json.loads(train_rows[0]["messages"][-1]["content"]).keys()))
print("user preview:\n", train_rows[0]["messages"][1]["content"][:240])

if len(train_rows) < 500:
    raise RuntimeError("train set looks too small — re-upload kaggle_dataset.zip from the Mac repo")


## 3) Load model + LoRA


In [ ]:
from unsloth import FastLanguageModel
import torch

# Loads Unsloth's public Qwen mirror — no Hugging Face token / login
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("LoRA ready (Unsloth)", "trainable params ~", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 4) Format chat examples


In [ ]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_ds = Dataset.from_list(train_rows).map(formatting_prompts_func, batched=True, remove_columns=["messages"])
valid_ds = Dataset.from_list(valid_rows).map(formatting_prompts_func, batched=True, remove_columns=["messages"])
print(train_ds)
print(train_ds[0]["text"][:400])


## 5) Train LoRA — keep tab open (~45–90 min on T4)


In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16

sft_args = dict(
    output_dir=str(WORK / "sft_out"),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    fp16=use_fp16,
    bf16=use_bf16,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

# Compatibility across TRL versions
params = set(inspect.signature(SFTConfig.__init__).parameters)
sft_args = {k: v for k, v in sft_args.items() if k in params}
# older aliases
if "evaluation_strategy" in params and "eval_strategy" not in params:
    sft_args["evaluation_strategy"] = "steps"

config = SFTConfig(**sft_args)

trainer_kwargs = dict(
    model=model,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    args=config,
)
trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters)
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
if "dataset_text_field" in trainer_params and "dataset_text_field" not in set(inspect.signature(SFTConfig.__init__).parameters):
    trainer_kwargs["dataset_text_field"] = "text"

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()

model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("adapters saved to", ADAPTER_DIR)


## 6) Smoke-test: must return JSON with code + output


In [ ]:
FastLanguageModel.for_inference(model)

probe = [
    {"role": "system", "content": train_rows[0]["messages"][0]["content"]},
    {"role": "user", "content": (
        "Course: CS-401 — Machine Learning Lab\n"
        "Lab: Lab 06 — Regression with Scikit-learn\n"
        "Task 1 — By hand\n"
        "For points (1,2),(2,3),(3,5),(4,4),(5,6) compute slope/intercept/SSE/R2 with formulas "
        "and sklearn; save a fit plot to plot.png.\n\n"
        "Write a complete Python solution for this single task. "
        "Return JSON only with keys code and output. No placeholders, no markdown."
    )},
]

prompt = tokenizer.apply_chat_template(probe, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=1024, temperature=0.1, do_sample=False)
text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(text[:1500])

# Soft parse check
import re
m = re.search(r"\{[\s\S]*\}", text)
assert m, "Model did not return a JSON object"
obj = json.loads(m.group(0))
assert "code" in obj and "output" in obj, obj.keys()
assert len(obj["code"].strip()) > 40, "code too short"
assert "placeholder" not in obj["code"].lower()
print("SMOKE OK — keys", sorted(obj.keys()), "code_chars", len(obj["code"]))


## 7) Merge + export GGUF Q4_K_M


In [ ]:
import shutil
from pathlib import Path

GGUF_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

def find_q4_gguf(root: Path) -> Path | None:
    """Unsloth may write Q4 under gguf/, gguf_gguf/, or nested model folders."""
    cands = []
    for p in root.rglob("*.gguf"):
        if not p.is_file():
            continue
        name = p.name.lower()
        # Prefer Q4_K_M; skip f16 intermediates
        if "f16" in name or "fp16" in name:
            continue
        if "q4_k_m" in name or "q4_k" in name:
            cands.append(p)
    if not cands:
        # any non-f16 gguf as last resort
        cands = [
            p for p in root.rglob("*.gguf")
            if p.is_file() and "f16" not in p.name.lower()
            and p.stat().st_size > 50_000_000
        ]
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

# --- Unsloth merge (optional but useful backup) ---
print("Merging LoRA with Unsloth (merged_16bit)...")
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")
print("merged →", MERGED_DIR)

# --- Unsloth GGUF Q4_K_M only (no HF push, no llama.cpp) ---
print("Exporting GGUF Q4_K_M with Unsloth...")
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

found = find_q4_gguf(WORK)
if found is None:
    found = find_q4_gguf(Path("/kaggle/working"))

if found is None:
    raise FileNotFoundError(
        "Unsloth did not produce a Q4_K_M .gguf under /kaggle/working/labpulse. "
        "Re-run this cell with Internet ON. Do not use Hugging Face login."
    )

if found.resolve() != GGUF_OUT.resolve():
    GGUF_OUT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(found, GGUF_OUT)
    print("copied", found.name, "→", GGUF_OUT.name)

assert GGUF_OUT.exists() and GGUF_OUT.stat().st_size > 50_000_000, GGUF_OUT
print("READY (Unsloth):", GGUF_OUT)
print("size_MB=", round(GGUF_OUT.stat().st_size / 1024 / 1024, 1))
print("DOWNLOAD THIS FILE NOW — stopping the session deletes /kaggle/working")


## 8) Download (do this before stopping the session)

Sidebar → **Output** → `labpulse` → `gguf` → **`labpulse-1.5b-q4_k_m.gguf`** only.

Or run the next cell for a one-click `FileLink`.

```bash
mv ~/Downloads/labpulse-1.5b-q4_k_m.gguf /Users/Macbook/Labpulse/models/labpulse-1.5b-q4_k_m.gguf
cd /Users/Macbook/Labpulse
GGML_METAL=0 LABPULSE_N_GPU_LAYERS=0 .venv/bin/uvicorn src.api:app --host 127.0.0.1 --port 8080
```


In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
import shutil

GGUF_OUT = Path("/kaggle/working/labpulse/gguf/labpulse-1.5b-q4_k_m.gguf")
print("GGUF:", GGUF_OUT, "exists=", GGUF_OUT.exists())
if not GGUF_OUT.exists():
    # recover if Unsloth wrote under a different name
    cands = [
        p for p in Path("/kaggle/working/labpulse").rglob("*.gguf")
        if p.is_file() and "f16" not in p.name.lower() and p.stat().st_size > 50_000_000
    ]
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    if cands:
        GGUF_OUT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cands[0], GGUF_OUT)
        print("recovered from", cands[0])

if GGUF_OUT.exists():
    print("size_MB=", round(GGUF_OUT.stat().st_size / 1024 / 1024, 1))
    # easy download at /kaggle/working root
    link = Path("/kaggle/working") / GGUF_OUT.name
    if not link.exists():
        shutil.copy2(GGUF_OUT, link)
    display(FileLink(str(link.name)))
else:
    print("MISSING — re-run merge/GGUF cell (Unsloth only). Do not stop the session.")


## Troubleshooting

| Issue | Fix |
|-------|-----|
| No GPU | Settings → Accelerator → **GPU** |
| Internet / DNS errors | Settings → Internet **ON** (Unsloth needs it once for the base model) |
| HF token / login prompts | Ignore — this notebook does **not** use Hugging Face auth. Re-run Config cell. |
| CUDA OOM | `USE_3B=False`, `BATCH_SIZE=1`, `MAX_SEQ_LENGTH=3072` |
| Dataset not found | Add `kaggle_dataset.zip` via **Add Data** |
| Smoke test fails | Confirm dataset is the new code/output zip; keep `EPOCHS=3` |
| GGUF missing after export | Re-run cell 7 only; check Output for `*Q4_K_M*.gguf` |
| Session stopped | `/kaggle/working` is wiped — must **Run All** again |

## After swap on Mac
Generate ML Lab 06 / OS Lab 08 again — expect real sklearn/`savefig` or OS `fork`/`exec`, not placeholders.
